# Train/val/test split + augmentation

Same split gets used by both the LSTM and CNN tracks so the two models are actually comparable at the end.

Split: 70% train, 15% val, 15% test, stratified by composer so each split has a similar composer mix.

Then augmentation, only on the training set (never touch val/test, that would leak info and make our numbers fake). Bach has way more files than the other 3 composers (1024 vs 136 for Chopin), so we pitch-shift the minority composers to boost their training counts. Val/test stay untouched and unbalanced since that's what the model will actually see in the real world.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

LSTM_DIR = Path("../data/processed/lstm")
CNN_DIR = Path("../data/processed/cnn")
SPLIT_DIR = Path("../data/splits")
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

lstm_manifest = pd.read_csv("../data/processed/lstm_manifest.csv")
cnn_manifest = pd.read_csv("../data/processed/cnn_manifest.csv")
print("lstm files:", len(lstm_manifest), "cnn files:", len(cnn_manifest))

lstm files: 1635 cnn files: 1635


In [2]:
# only keep files that made it through BOTH pipelines, should basically be everything
# minus the 2 corrupt files but check just in case
lstm_keys = set(zip(lstm_manifest.composer, lstm_manifest.filename))
cnn_keys = set(zip(cnn_manifest.composer, cnn_manifest.filename))
both = lstm_keys & cnn_keys
print("in both:", len(both), "lstm only:", len(lstm_keys - cnn_keys), "cnn only:", len(cnn_keys - lstm_keys))

files_df = pd.DataFrame(list(both), columns=["composer", "filename"])
files_df["composer"].value_counts()

in both: 1635 lstm only: 0 cnn only: 0


composer
bach         1024
mozart        256
beethoven     219
chopin        136
Name: count, dtype: int64

In [3]:
# 70/15/15 stratified split. do it in 2 steps: first split off train, then split
# the leftover 30% into val/test evenly
train_df, temp_df = train_test_split(
    files_df, test_size=0.30, stratify=files_df["composer"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["composer"], random_state=42
)

print("train:", len(train_df), "val:", len(val_df), "test:", len(test_df))
train_df["composer"].value_counts()

train: 1144 val: 245 test: 246


composer
bach         717
mozart       179
beethoven    153
chopin        95
Name: count, dtype: int64

In [4]:
val_df.to_csv(SPLIT_DIR / "val.csv", index=False)
test_df.to_csv(SPLIT_DIR / "test.csv", index=False)
train_df.to_csv(SPLIT_DIR / "train_original.csv", index=False)  # unaugmented train list, keep for reference

## Augmentation (training set only)

Pitch shift every note by N semitones. Cheap and works on both the LSTM sequences (just add N to the pitch array) and the CNN piano rolls (shift the rows up/down, zero fill whatever falls off the edge).

Shift amounts per composer, picked to roughly close the gap with Bach without going overboard:
- bach: none, already has plenty
- beethoven: shift by -1 and +1 (3x total)
- mozart: shift by -1 and +1 (3x total)
- chopin: shift by -2, -1, +1, +2 (5x total, needs the most help)

In [5]:
SHIFTS = {
    "bach": [],
    "beethoven": [-1, 1],
    "mozart": [-1, 1],
    "chopin": [-2, -1, 1, 2],
}

In [6]:
def augment_lstm_file(npz_path, shift, out_path):
    d = np.load(npz_path)
    pitch = np.clip(d["pitch"].astype(np.int16) + shift, 0, 127).astype(np.int16)
    np.savez_compressed(out_path, pitch=pitch, duration=d["duration"], velocity=d["velocity"], offset=d["offset"])


def augment_cnn_file(npy_path, shift, out_path):
    roll = np.load(npy_path)
    shifted = np.zeros_like(roll)
    if shift > 0:
        shifted[shift:, :] = roll[: roll.shape[0] - shift, :]
    elif shift < 0:
        shifted[: roll.shape[0] + shift, :] = roll[-shift:, :]
    else:
        shifted = roll.copy()
    np.save(out_path, shifted)

In [7]:
# quick test before running on the whole train set
row = train_df.iloc[0]
stem = Path(row.filename).stem
test_lstm_out = LSTM_DIR / row.composer / f"{stem}_shiftTEST.npz"
test_cnn_out = CNN_DIR / row.composer / f"{stem}_shiftTEST.npy"

augment_lstm_file(LSTM_DIR / row.composer / f"{stem}.npz", 2, test_lstm_out)
augment_cnn_file(CNN_DIR / row.composer / f"{stem}.npy", 2, test_cnn_out)

orig_pitch = np.load(LSTM_DIR / row.composer / f"{stem}.npz")["pitch"][:5]
shifted_pitch = np.load(test_lstm_out)["pitch"][:5]
print("orig pitch:", orig_pitch)
print("shifted +2:", shifted_pitch)
assert np.array_equal(orig_pitch + 2, shifted_pitch)

print("cnn shapes match:", np.load(test_cnn_out).shape == np.load(CNN_DIR / row.composer / f"{stem}.npy").shape)

# clean up the test files
test_lstm_out.unlink()
test_cnn_out.unlink()
print("looks good")

orig pitch: [79 80 82 67 80]
shifted +2: [81 82 84 69 82]
cnn shapes match: True
looks good


In [8]:
augmented_rows = []

# start with the original train files, not augmented
for _, row in train_df.iterrows():
    augmented_rows.append({
        "composer": row.composer, "filename": row.filename,
        "source_filename": row.filename, "shift": 0, "is_augmented": False,
    })

# now add the shifted copies
for _, row in train_df.iterrows():
    shifts = SHIFTS.get(row.composer, [])
    stem = Path(row.filename).stem

    for shift in shifts:
        tag = f"shift{shift:+d}".replace("+", "p").replace("-", "m")
        new_filename = f"{stem}_{tag}.mid"  # keeping .mid suffix so it still lines up with the naming convention, even though there's no actual midi file for this one

        lstm_src = LSTM_DIR / row.composer / f"{stem}.npz"
        lstm_dst = LSTM_DIR / row.composer / f"{stem}_{tag}.npz"
        augment_lstm_file(lstm_src, shift, lstm_dst)

        cnn_src = CNN_DIR / row.composer / f"{stem}.npy"
        cnn_dst = CNN_DIR / row.composer / f"{stem}_{tag}.npy"
        augment_cnn_file(cnn_src, shift, cnn_dst)

        augmented_rows.append({
            "composer": row.composer, "filename": new_filename,
            "source_filename": row.filename, "shift": shift, "is_augmented": True,
        })

train_aug_df = pd.DataFrame(augmented_rows)
print("train rows before augmentation:", len(train_df))
print("train rows after augmentation:", len(train_aug_df))

train rows before augmentation: 1144
train rows after augmentation: 2188


In [9]:
train_aug_df.to_csv(SPLIT_DIR / "train.csv", index=False)

print("class balance before augmentation:")
print(train_df["composer"].value_counts())
print()
print("class balance after augmentation:")
print(train_aug_df["composer"].value_counts())

class balance before augmentation:
composer
bach         717
mozart       179
beethoven    153
chopin        95
Name: count, dtype: int64

class balance after augmentation:
composer
bach         717
mozart       537
chopin       475
beethoven    459
Name: count, dtype: int64


## Notes

- `data/splits/train.csv` is the one to actually use for training, it has the augmented rows mixed in with `is_augmented` flag so we can tell them apart later if needed
- `data/splits/train_original.csv` is just the pre-augmentation list, kept around in case we need to compare with/without augmentation later
- val.csv and test.csv are untouched, no augmentation, real world distribution
- still not perfectly balanced after augmentation (bach still has way more than chopin) but it's a lot closer than before, might need class weights on top of this when we actually train the models